# Dense Retrieval & Vector Search with FAISS and Sentence-Transformers

This notebook demonstrates the practical mechanics of Dense Retrieval using real pre-trained transformer embeddings (sentence-transformers) and an industry-standard vector search index (FAISS).   

It covers:



*   Generating real high-dimensional embeddings ($384$ dimensions).   
*   Indexing dense vectors using Euclidean (L2), Inner Product (Dot Product), and Cosine Similarity metrics.
*   Demonstrating semantic search (retrieval without exact keyword matching).
*   Comparing how vector magnitude impacts Dot Product vs. Cosine Similarity vs. L2 Distance rankings.




  
     

In [ ]:
!pip install -q sentence-transformers faiss-cpu numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.6 MB/s eta 0:00:00


# Load Document Corpus
We use a practical technical support knowledge base corpus. Notice that documents use different phrasing and synonyms for common issue types.

In [ ]:
import pandas as pd

# Technical support knowledge base
corpus = [
    {"doc_id": "D1", "category": "Account", "text": "To restore access when locked out, follow the identity verification process on the recovery portal."},
    {"doc_id": "D2", "category": "Account", "text": "You can update your display name, profile avatar, and primary contact details from your profile preferences."},
    {"doc_id": "D3", "category": "Billing", "text": "View, download, and pay outstanding invoices or update credit card information in the billing dashboard."},
    {"doc_id": "D4", "category": "Hardware", "text": "If your laptop monitor stays blank upon powering up, try performing a hard reset by holding the power key for 15 seconds."},
    {"doc_id": "D5", "category": "Network", "text": "Improve local wireless signals by restarting your router and repositioning it away from large metal objects."},
    {"doc_id": "D6", "category": "Security", "text": "Enable multi-factor authentication (MFA) to require an authenticator app code during sign-in attempts."},
    {"doc_id": "D7", "category": "Account", "text": "If you forgot your login password, click 'Forgot Password' on the sign-in screen to receive a reset link via email."}
]

df_corpus = pd.DataFrame(corpus)
print(f"Loaded {len(df_corpus)} documents into corpus.")
df_corpus[['doc_id', 'category', 'text']]

Loaded 7 documents into corpus.


,doc_id,category,text
0,D1,Account,"To restore access when locked out, follow the ..."
1,D2,Account,"You can update your display name, profile avat..."
2,D3,Billing,"View, download, and pay outstanding invoices o..."
3,D4,Hardware,If your laptop monitor stays blank upon poweri...
4,D5,Network,Improve local wireless signals by restarting y...
5,D6,Security,Enable multi-factor authentication (MFA) to re...
6,D7,Account,"If you forgot your login password, click 'Forg..."


#Generate Dense Vector Embeddings
We use the all-MiniLM-L6-v2 transformer model to encode unstructured text into $384$-dimensional dense vector space.   

In [ ]:
from sentence_transformers import SentenceTransformer

# Load pre-trained embedding model
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

# Extract plain text documents
documents = df_corpus['text'].tolist()

# Generate dense embeddings (returns numpy ndarray: num_docs x vector_dim)
doc_embeddings = model.encode(documents, convert_to_numpy=True, show_progress_bar=True)

# FAISS requires 32-bit float data types
doc_embeddings = doc_embeddings.astype('float32')

print(f"\nEmbedding Matrix Shape: {doc_embeddings.shape}")
print(f"Vector Dimension (d): {doc_embeddings.shape[1]}")
print(f"Sample values of Document D1 (first 5 dimensions):\n{doc_embeddings[0][:5].round(4)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding Matrix Shape: (7, 384)
Vector Dimension (d): 384
Sample values of Document D1 (first 5 dimensions):
[-0.0409  0.038  -0.0089  0.019   0.0548]


#FAISS Vector Indexing
The embedding model maps semantics to continuous vector space, while the vector engine (FAISS) indexes and searches vectors efficiently.   

We set up three indices:


*   L2 (Euclidean) Distance (faiss.IndexFlatL2): Measures spatial distance between endpoints.
*   Inner Product / Dot Product (faiss.IndexFlatIP): Unnormalized similarity affected by vector magnitude.
*   Cosine Similarity: Inner product performed on L2-normalized vectors ($\Vert{}v\Vert{}_2 = 1.0$).  



  

In [ ]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1] # 384 dimensions

# 1. FAISS Index for Euclidean Distance (L2)
index_l2 = faiss.IndexFlatL2(dimension)
index_l2.add(doc_embeddings)

# 2. FAISS Index for Unnormalized Dot Product (Inner Product)
index_ip = faiss.IndexFlatIP(dimension)
index_ip.add(doc_embeddings)

# 3. FAISS Index for Cosine Similarity (Normalized Inner Product)
doc_embeddings_norm = doc_embeddings.copy()
faiss.normalize_L2(doc_embeddings_norm)  # Scales each vector length to 1.0

index_cosine = faiss.IndexFlatIP(dimension)
index_cosine.add(doc_embeddings_norm)

print(f"Successfully indexed {index_cosine.ntotal} document vectors across 3 FAISS index types.")

Successfully indexed 7 document vectors across 3 FAISS index types.


#Execute Semantic Search
Dense retrieval identifies relevant documents even when the query shares zero exact terms with the candidate documents.

In [ ]:
# Query uses natural phrasing ("can't sign in", "lost my password")
query_text = "I am unable to sign into my profile because I lost my password"

# Encode query with the same transformer model
query_embedding = model.encode([query_text], convert_to_numpy=True).astype('float32')

# Prepare normalized query vector for Cosine Similarity
query_embedding_norm = query_embedding.copy()
faiss.normalize_L2(query_embedding_norm)

# Search top K=3 matches using Cosine Similarity
k = 3
scores_cosine, indices_cosine = index_cosine.search(query_embedding_norm, k)

print(f"QUERY: '{query_text}'\n")
print("Top-3 Retrieved Documents (Cosine Similarity):")
print("=" * 75)

for rank, (idx, score) in enumerate(zip(indices_cosine[0], scores_cosine[0]), start=1):
    matched_doc = df_corpus.iloc[idx]
    print(f"Rank {rank} | Cosine Similarity: {score:.4f} | ID: {matched_doc['doc_id']}")
    print(f"Content: \"{matched_doc['text']}\"\n")

QUERY: 'I am unable to sign into my profile because I lost my password'

Top-3 Retrieved Documents (Cosine Similarity):
Rank 1 | Cosine Similarity: 0.5912 | ID: D7
Content: "If you forgot your login password, click 'Forgot Password' on the sign-in screen to receive a reset link via email."

Rank 2 | Cosine Similarity: 0.5006 | ID: D1
Content: "To restore access when locked out, follow the identity verification process on the recovery portal."

Rank 3 | Cosine Similarity: 0.3390 | ID: D6
Content: "Enable multi-factor authentication (MFA) to require an authenticator app code during sign-in attempts."



#Compare Ranking Differences (Dot Product vs. Cosine vs. L2)

Vector magnitude $\Vert{}d\Vert{}$ influences raw dot products ($q \cdot d$), whereas cosine similarity isolates directional alignment:

In [ ]:
k = len(df_corpus) # Retrieve all documents to compare full ranking orders

# Retrieve rankings across all three metrics
scores_cos, idx_cos = index_cosine.search(query_embedding_norm, k)
scores_dot, idx_dot = index_ip.search(query_embedding, k)
scores_l2, idx_l2 = index_l2.search(query_embedding, k)

comparison_rows = []
for r in range(k):
    doc_cos = df_corpus.iloc[idx_cos[0][r]]['doc_id']
    doc_dot = df_corpus.iloc[idx_dot[0][r]]['doc_id']
    doc_l2 = df_corpus.iloc[idx_l2[0][r]]['doc_id']

    comparison_rows.append({
        "Rank": r + 1,
        "Cosine Similarity Order": f"{doc_cos} ({scores_cos[0][r]:.4f})",
        "Raw Dot Product Order": f"{doc_dot} ({scores_dot[0][r]:.4f})",
        "L2 Distance Order": f"{doc_l2} (Dist: {scores_l2[0][r]:.4f})"
    })

df_comparison = pd.DataFrame(comparison_rows)
df_comparison

,Rank,Cosine Similarity Order,Raw Dot Product Order,L2 Distance Order
0,1,D7 (0.5912),D7 (0.5912),D7 (Dist: 0.8176)
1,2,D1 (0.5006),D1 (0.5006),D1 (Dist: 0.9989)
2,3,D6 (0.3390),D6 (0.3390),D6 (Dist: 1.3221)
3,4,D2 (0.2645),D2 (0.2645),D2 (Dist: 1.4711)
4,5,D4 (0.1350),D4 (0.1350),D4 (Dist: 1.7300)
5,6,D3 (0.0249),D3 (0.0249),D3 (Dist: 1.9502)
6,7,D5 (-0.0336),D5 (-0.0336),D5 (Dist: 2.0672)


$$
\begin{array}{|l|l|}
\hline
\textbf{Concept} & \textbf{Description} \\ \hline
\text{Embedding Model vs. Vector Retrieval} & \begin{array}{l} \text{The embedding model translates textual semantics into multi-dimensional vectors.} \\ \text{The vector index (FAISS) performs high-speed nearest-neighbor vector comparison and ranking.} \end{array} \\ \hline
\text{Semantic Matching} & \text{Matches documents based on conceptual meaning rather than exact string or keyword token overlap.} \\ \hline
\text{Metric Effects on Ranking} & \begin{array}{l} \text{Unnormalized Dot Product } (q \cdot d) \text{ scales with vector magnitude. Normalizing vectors to unit length } (\|v\|_2 = 1.0) \\ \text{transforms Inner Product into pure Cosine Similarity, preventing long or highly dense vectors} \\ \text{from dominating rankings due to magnitude alone.} \end{array} \\ \hline
\end{array}
$$